# Customer Retention & Churn Prediction Model
**Domain:** Telecommunications & Subscription Marketing  
**Methodology:** Binary Classification, Predictive Modeling, Feature Importance Analysis  

---

## 1. Introduction & Project Overview

In the highly competitive telecommunications industry, customer acquisition costs (CAC) drastically exceed customer retention costs. Acquiring a new subscriber can cost up to 5 times more than keeping an existing one satisfied. For a telecom company, predicting customer churn before it happens is the single most effective way to protect recurring revenue streams and stabilize profit margins.

The objective of this project is to build a robust machine learning pipeline using Python and Scikit-Learn to accurately predict which customers are at risk of canceling their service (`Churn = 1`). Furthermore, by extracting mathematical feature importances from our model, we reveal the core behavioral triggers behind churn, allowing growth and marketing teams to launch automated, proactive retention campaigns before customers leave.

---

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("barun2104/telecom-churn")

print("Path to dataset files:", path)

100%|██████████| 45.5k/45.5k [00:00<00:00, 32.2MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/barun2104/telecom-churn/versions/2


In [7]:
import os

df = pd.read_csv(os.path.join(path, 'telecom_churn.csv'))

# 2. Separate target variable (Churn) from predictors (Features)
X = df.drop('Churn', axis=1)
y = df['Churn']

# 3. Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Scale continuous features so large numbers don't distort the model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train a Random Forest Classifier
# class_weight='balanced' handles the fact that only ~14% of your users churned
model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# 6. Evaluate model accuracy and precision
y_pred = model.predict(X_test_scaled)
print("================ MODEL PERFORMANCE ================")
print(classification_report(y_test, y_pred))

# 7. Extract actionable marketing insights (Feature Importance)
importances = model.feature_importances_
feature_imp_df = pd.DataFrame({'Metric': X.columns, 'Importance_Score': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance_Score', ascending=False)

print("\n============ DRIVERS OF CUSTOMER CHURN ============")
print(feature_imp_df.to_string(index=False))

================ MODEL PERFORMANCE ================
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       570
           1       0.81      0.62      0.70        97

    accuracy                           0.92       667
   macro avg       0.87      0.80      0.83       667
weighted avg       0.92      0.92      0.92       667


============ DRIVERS OF CUSTOMER CHURN ============
         Metric  Importance_Score
        DayMins          0.181057
  CustServCalls          0.177938
  MonthlyCharge          0.164244
ContractRenewal          0.104393
     OverageFee          0.085764
      DataUsage          0.074418
       RoamMins          0.068524
       DayCalls          0.063160
   AccountWeeks          0.060817
       DataPlan          0.019685


### Deconstructing the Performance:
*   **92% Overall Accuracy:** The model correctly classifies 92 out of every 100 customers in a completely unseen evaluation set.
*   **81% Churn Precision:** When the model flags a customer as a churn risk, it is mathematically correct **81% of the time**. This ensures marketing budgets are spent exclusively on high-risk users rather than being wasted on safe accounts.
*   **62% Churn Recall (F1-Score: 0.70):** The model actively intercepts **62% of all actual churners** before they exit. An F1-score of 0.70 represents a highly reliable, production-ready benchmark for behavioral churn tasks.

### Deep-Dive Analysis:
1.  **DayMins (18.1%):** High daytime call volume is the single largest indicator of churn. Heavy communicators rely heavily on network reliability and plan flexibility, making them highly volatile if expectations aren't met.
2.  **CustServCalls (17.7%):** Every call to support is a point of customer friction. Repeated calls indicate unresolved, compounding technical or billing issues.
3.  **MonthlyCharge (16.4%):** Customers are explicitly price-sensitive. Elevated charges without clear utility drive users to aggressively compare market competitors.

---

## 5. Actionable Marketing & Retention Strategies

To translate this machine learning model into direct business revenue, we propose three automated marketing campaigns based on our findings:

*   **Campaign 1: The '3-Call' Proactive Support Trigger**
    *   *Insight:* `CustServCalls` is a major churn catalyst.
    *   *Action:* Establish an automated workflow in the CRM. The moment any account hits **3 or more customer service calls** within a billing cycle, they are automatically placed on a high-priority retention queue. A dedicated support specialist proactively contacts them to resolve their core pain points before they decide to cancel.
*   **Campaign 2: High-Volume Usage Plan Optimization**
    *   *Insight:* High `DayMins` coupled with high `MonthlyCharge` creates an immediate churn risk.
    *   *Action:* Use the model to scan for heavy daytime talkers getting hit with high bills or `OverageFee` spikes. Have marketing automatically message them: *"We noticed your high daytime usage! Let’s move you to our Unlimited High-Volume Plan, which will actually save you $10 a month."* This builds intense brand loyalty and eliminates the price shock.
*   **Campaign 3: Pre-Expiration Lock-In Rewards**
    *   *Insight:* `ContractRenewal` accounts for 10.4% of churn weight.
    *   *Action:* Identify accounts nearing their contract conclusion (`ContractRenewal = 0`). Target these accounts 30 days prior to expiration with automated contract-extension incentives, such as a loyalty credit or a data plan upgrade, locking them in for another 12-month cycle.

#Author

Christian Bisrat